In [1]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import numpy as np 
import pandas as pd
import joblib

ModuleNotFoundError: No module named 'numpy'

In [ ]:
df_name = "crop_drought_resistance_prediction.csv"
df = pd.read_csv(df_name)

print(df.shape)
df.info()

In [ ]:
df = df.drop(columns=["Latitude", "Longitude", "Date"])
df.drop(columns=["Drought_Score"]).columns

In [ ]:
strings = df.select_dtypes(include=["object"]).columns
encoders = dict()

for col in strings:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[[col]])
    encoders[col] = encoder
    
df = df.astype(np.float32)

df.describe()

In [ ]:
for key, value in encoders.items():
    print(f"Classes for {key}")
    classes = list(value.classes_)
    for i in classes:
        print(f"{i}: {value.transform([i])[0]}")
    print("")

In [ ]:
model = RandomForestRegressor(random_state=42)  
params = {
    "max_features": [i for i in range(1, 10)],
    "n_estimators": [i for i in range(90, 100)]
}

x = df.drop(columns=[target])
y = df[target]

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)
grid = GridSearchCV(model, param_grid=params, cv=3)
grid.fit(xtrain, ytrain)

In [ ]:
print(f"[+] Best CV Result: {grid.best_score_*100}%")
best_model = grid.best_estimator_
y_train_preds = best_model.predict(xtrain)
y_test_preds = best_model.predict(xtest)

print("[+] Train Data Accuracy:", f"{r2_score(ytrain, y_train_preds)*100}%")
print("[+] Test Data Accuracy:", f"{r2_score(ytest, y_test_preds)*100}%")

In [ ]:
joblib.dump(encoders, "./Models/encoders.pkl")
joblib.dump(model, "./Models/model.pkl")

In [ ]:
for index, param in enumerate(grid.cv_results_["params"]):
    max_features = param['max_features']
    n_estimators = param['n_estimators']
    print(f"[+] Params: Max Features: {max_features}, n Estimators: {n_estimators}")
    print(f'     Split 1 Accuracy: {grid.cv_results_["split0_test_score"][index]*100}%')
    print(f'     Split 2 Accuracy: {grid.cv_results_["split1_test_score"][index]*100}%')
    print(f'     Split 3 Accuracy: {grid.cv_results_["split2_test_score"][index]*100}%')
    print("")
    